# 🔴 Taible — AMD ROCm AI Pipeline
## Team: team-3719 | Hackathon GPU Notebook

This notebook:
1. Verifies AMD GPU and ROCm setup
2. Starts **vLLM** with **Qwen2.5-7B-Instruct** on AMD GPU
3. Creates a public tunnel URL so your local Pipecat connects to AMD GPU

## Step 1: Verify AMD GPU

In [ ]:
# Verify AMD GPU and ROCm environment
import subprocess

print('=== AMD GPU Info ===')
result = subprocess.run(['rocm-smi', '--showproductname'], capture_output=True, text=True)
print(result.stdout)

print('\n=== ROCm Version ===')
result = subprocess.run(['rocminfo'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if 'Marketing' in line or 'Name' in line or 'ROCm' in line:
        print(line)

print('\n=== PyTorch AMD Check ===')
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'ROCm available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {mem:.1f} GB')

## Step 2: Install cloudflared for public tunnel safely (Bypass SSL Proxy)

In [ ]:
import urllib.request, os, ssl

# AMD Hackathon environments sometimes use internal proxies with self-signed certs
ssl._create_default_https_context = ssl._create_unverified_context

url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
print(f"Downloading cloudflared from {url} ...")

class RedirectHandler(urllib.request.HTTPRedirectHandler):
    def http_error_302(self, req, fp, code, msg, headers):
        return urllib.request.HTTPRedirectHandler.http_error_302(self, req, fp, code, msg, headers)

opener = urllib.request.build_opener(RedirectHandler())
urllib.request.install_opener(opener)

urllib.request.urlretrieve(url, "/tmp/cloudflared")
os.chmod("/tmp/cloudflared", 0o755)

# Check if it is actually an ELF binary
with open("/tmp/cloudflared", "rb") as f:
    header = f.read(4)
    if header != b'\x7fELF':
        print("\n❌ Error: Downloaded file is NOT a valid binary. GitHub might be blocking the download.")
        print("File header:", header)
    else:
        print("\n✅ cloudflared installed successfully and is a valid binary!")


## Step 3: Start vLLM Server on AMD GPU

In [ ]:
import subprocess, time, requests, threading, sys

VLLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
VLLM_PORT  = 8000

# Using 0.5 utilization to avoid out-of-memory if the GPU has other processes running
vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', VLLM_MODEL,
    '--port', str(VLLM_PORT),
    '--dtype', 'auto',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.50',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
    '--served-model-name', 'Qwen/Qwen2.5-7B-Instruct',
]

print(f'Starting vLLM with {VLLM_MODEL} on AMD GPU...')
print('This may take 2-3 minutes to download and load the model.')
print('='*60)

# Fallback: if vllm is not in this python env, let's install it first
try:
    import vllm
except ImportError:
    print("Installing vllm...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'vllm'], capture_output=True)

vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream logs until server is ready
ready = False
for line in vllm_proc.stdout:
    print(line, end='')
    if 'Application startup complete' in line or 'Uvicorn running' in line:
        ready = True
        print('\n✅ vLLM Server is READY on AMD GPU!')
        break

if not ready:
    print('Waiting for health check...')
    time.sleep(10)

## Step 4: Verify vLLM is running

In [ ]:
import requests, json

try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    print(f'✅ vLLM Health: {r.status_code} OK')
    
    r2 = requests.get('http://localhost:8000/v1/models', timeout=5)
    models = r2.json()
    print(f'\n📦 Loaded Models:')
    for m in models.get('data', []):
        print(f"  - {m['id']}")
except Exception as e:
    print(f'❌ vLLM not ready yet: {e}')
    print('Run the previous cell again or wait 30 seconds.')

## Step 5: Test AMD GPU inference

In [ ]:
import requests, json

payload = {
    'model': 'Qwen/Qwen2.5-7B-Instruct',
    'messages': [
        {'role': 'system', 'content': 'You are Alma, a friendly waiter at Cafe Alma. Reply in 1 sentence.'},
        {'role': 'user', 'content': 'Hi! What do you have on the menu today?'}
    ],
    'max_tokens': 100,
    'temperature': 0.7
}

try:
    r = requests.post('http://localhost:8000/v1/chat/completions', json=payload, timeout=30)
    result = r.json()

    print('🤖 AMD GPU (Qwen2.5-7B) Response:')
    print(result['choices'][0]['message']['content'])
    print(f"\n📊 Token Usage:")
    print(f"  Prompt tokens: {result['usage']['prompt_tokens']}")
    print(f"  Completion tokens: {result['usage']['completion_tokens']}")
except Exception as e:
    print(f"Test failed: {e}")

## Step 6: Create Public Tunnel URL
### This gives your local Pipecat the AMD GPU URL

In [ ]:
import subprocess, threading, time, re

tunnel_url = None

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end='')
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print(f'\n')
            print('='*60)
            print(f'🚀 AMD GPU PUBLIC URL:')
            print(f'   {tunnel_url}')
            print('='*60)
            print(f'\n👆 Copy this URL and set it in your .env file:')
            print(f'   VLLM_BASE_URL={tunnel_url}')
            break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

# Wait up to 30 seconds for URL
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(1)

if not tunnel_url:
    print('⏳ Still waiting for tunnel... check output above.')

## Step 7: Keep notebook alive (run in background)
### The tunnel will stay active as long as this cell is running

In [ ]:
import time

print('✅ AMD GPU pipeline is ACTIVE!')
print('='*60)
print(f'vLLM URL (local):  http://localhost:8000')
if tunnel_url:
    print(f'vLLM URL (public): {tunnel_url}')
print('='*60)
print('\n📋 Next Steps (on your LOCAL machine):')
print('1. Open voice-orchestration/.env')
print(f'2. Add: VLLM_BASE_URL={tunnel_url}')
print('3. Restart: python main.py')
print('\n⚠️  Keep this notebook running! Closing stops the AMD pipeline.')
print('\nHeartbeat (press Stop to end):')

counter = 0
while True:
    counter += 1
    print(f'  [{counter}] AMD GPU pipeline alive... {time.strftime("%H:%M:%S")}', end='\r')
    time.sleep(30)